# 06 · Bench — 05 Citation verification and the pass rubric

**Ported from `specialist-rag`'s `tests/benchmark/citation_check.py` and `tests/benchmark/pass_rubric.py`, branch `origin/fix-completion` at commit `130ff868` (not on `main`). Both now live alongside `eval.py` in `04-benchmarks/clinical-retrieval/` and are wired into every run -- this notebook demonstrates what they actually catch.**

Before this, `eval.py` pulled PMCIDs and DOIs out of an answer and did
nothing with them -- an answer citing a plausible-looking but invented DOI
passed. And the cookbook had no rubric at all: nothing defined the
threshold a run has to clear to count as a pass. Both gaps are now closed
in `eval.py` itself; this notebook shows the two modules that closed them,
in isolation, with real fabricated-citation and real threshold examples.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `verify_citations` | Checks every PMCID/DOI an answer cites against what was actually retrieved | `verify_citations(answer, papers)` |
| `evaluate_retrieval_pass` | Five named checks that together decide whether retrieval passed | `evaluate_retrieval_pass(result, case)` |
| `evaluate_citation_pass` | The rubric's citation layer -- delegates to `verify_citations`, doesn't re-implement it | `evaluate_citation_pass(answer, papers)` |
| `evaluate_question_pass` | Combines retrieval + citation + (optional) DeepEval into one verdict | `evaluate_question_pass(result, case, answer=answer)` |


In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(6):
    if (_p / "nbio.py").is_file():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")

import nbio

repo_root = nbio.bootstrap()

BENCH_DIR = repo_root / "04-benchmarks" / "clinical-retrieval"
sys.path.insert(0, str(BENCH_DIR))

## Step 1 — one retrieved paper, one answer that cites both a real and a fabricated identifier

The paper carries a real PMCID and a real DOI. The answer correctly cites
the PMCID, then adds a second citation to a DOI that was never retrieved --
the exact shape of a plausible-looking but invented citation.

In [ ]:
papers = [
    {
        "title": "Early excision reduces mortality in major burns",
        "summary": "Early tangential excision within 48-72 hours reduces mortality.",
        "pmcid": "PMC123456",
        "doi": "10.1000/burns.123",
        "evidence_level": "Level II",
        "rcs_score": 7,
    }
]

answer_with_fabrication = (
    "Early excision reduces mortality (PMC123456). This is further supported "
    "by 10.9999/fabricated.doi, a large multicenter trial."
)

print("papers retrieved:", len(papers))
print("answer:", answer_with_fabrication)

## Step 2 — `verify_citations`: which citation is real, which is invented

`known_identifiers` collects the PMCIDs/DOIs actually in `papers`;
`verify_citations` checks the answer's own citations against that set.

In [ ]:
from citation_check import verify_citations

result = verify_citations(answer_with_fabrication, papers)
nbio.show_json(result)

assert result["verified_pmcids"] == ["PMC123456"]
assert result["hallucinated_dois"] == ["10.9999/fabricated.doi"]
assert result["citation_pass"] is False
print()
print("confirmed: the real PMCID verifies, the fabricated DOI is caught, citation_pass is False")

## Step 3 — a clean answer, for contrast

Same paper, an answer that cites only the real PMCID. `citation_pass`
should now be `True` -- the check isn't just suspicious of every citation,
only unverifiable ones.

In [ ]:
clean_answer = "Early excision reduces mortality (PMC123456)."
clean_result = verify_citations(clean_answer, papers)
print("clean answer, citation_pass:", clean_result["citation_pass"])
assert clean_result["citation_pass"] is True

## Step 4 — `evaluate_retrieval_pass`: five checks, one verdict

Ported from `pass_rubric.py`. Retrieval passes only if papers were
returned, at least one required keyword was found, the paper-overlap check
passes, a majority of RCS scores are >=5, and every paper has an evidence
level assigned. Demonstrated on a `result` dict shaped like what
`eval.py`'s `run_one_question` produces.

In [ ]:
from pass_rubric import evaluate_retrieval_pass

fake_result = {
    "papers": papers,
    "paper_count": len(papers),
    "keywords_found": ["early excision"],
    "error": None,
}
fake_case = {"id": "B01", "expected_papers": []}

retrieval_verdict = evaluate_retrieval_pass(fake_result, fake_case)
nbio.show_json(retrieval_verdict)

assert retrieval_verdict["retrieval_pass"] is True

## Step 5 — `evaluate_question_pass`: the combined verdict, and a real gap worth knowing about

Runs retrieval + citation together. **Caution, verified directly here, not
just asserted in the module's own comment:** the donor's own `overall_pass`
does not fold in `citation_pass`, even when it's computed and reported
right next to it. The fabricated-citation answer below produces
`citation_pass: False` and `overall_pass: True` in the same result --
carried across honestly rather than silently fixed, since changing what
"pass" means is a decision for whoever owns this rubric, not a port
detail.

In [ ]:
from pass_rubric import evaluate_question_pass

verdict = evaluate_question_pass(fake_result, fake_case, answer=answer_with_fabrication)
nbio.show_json(verdict)

assert verdict["citation"]["citation_pass"] is False
assert verdict["overall_pass"] is True, "this IS the gap -- overall_pass does not see citation_pass"
print()
print("confirmed: citation_pass=False and overall_pass=True can both be true in one result -- know this before reading overall_pass as a single number")

## Where this runs in the pipeline

Both modules are imported directly by `04-benchmarks/clinical-retrieval/eval.py`
-- every real run now writes `result["citation_verification"]` and
`result["rubric"]` per question, and `print_table` reports how many
questions pass the retrieval rubric and overall. Run
`python eval.py --query-id B01` from that directory to see it on a real
(zero-paper, by default) run.